# Обзор проекта

**Стартовая статья документации проекта Ztare AI Local KB MCP.** Цели, принципы, два режима работы с базой, вспомогательные цели, решения, статус и карта остальных статей. Документация проекта ведётся здесь, в базе знаний (в репозитории — только `README.md` с кратким описанием и запуском); правки — через MCP `kb` или в JupyterLab.

## Главная цель

**MCP-плагин единой базы знаний для Jupyter.** База знаний доступна ИИ через MCP, а человек может её читать и редактировать прямо в JupyterLab.

Один плагин (расширение Jupyter Server), два MCP-endpoint на порту Jupyter, авторизация по его токену:

| Режим | Endpoint | Назначение |
|---|---|---|
| 1. Использование | `/mcp/kb` | работа с базой; профили `lite` / `std` / `full` |
| 2. Поддержание | `/mcp/kb-admin` | обслуживание базы (индексатор) |

Разделение позволяет выдавать агентам доступ только к режиму 1, а обслуживание оставить человеку или отдельному агенту.

## Принципы

- **Источник истины — ноутбуки `.ipynb` из Markdown-ячеек** (одна секция = одна ячейка) в `jupyter_data/kb/`: области → темы → подтемы → статьи → секции. Их можно читать и править в JupyterLab без ИИ.
- **Служебных файлов нет, индексы производные.** Сервер строит их в памяти при старте из ноутбуков и метаданных секций и обновляет по ходу работы.
- Секция — единица записи и индексации; служебные данные — в метаданных секции (`metadata.kb`).
- Откат ошибок — через git (автокоммиты), черновиков нет.

## Режимы работы

### Режим 1: использование (`/mcp/kb`)
- **Поиск:** по тексту, заголовкам секций и тегам (позже — семантический по эмбеддингам). Результат — статьи и секции.
- **Чтение:** тема (шапка), статья (оглавление или целиком), секция; теги — по параметру.
- **Запись:** создать тему или статью с шапкой, создать или изменить секцию. Изменённая секция получает `indexed = false`.
- Секции без заголовка в этом режиме не видны.

### Режим 2: поддержание (`/mcp/kb-admin`)
- **Индексация:** секции с `indexed = false` (от старых к новым) — теги индексатора, обновление описаний в шапках. Правки из веба и мимо Jupyter находятся по хешу текста.
- **Наведение порядка:** заголовки секциям, шапки статьям, деление/объединение секций.
- **Агрегация:** сводки и обзорные (hub) заметки по темам (с помощью LLM).
- **Чистка:** дубликаты, битые ссылки, пустые и устаревшие заметки — сначала отчёт, применение после подтверждения.

## Вспомогательные цели и рамки

- **Локальная LLM на NPU Galaxy S26 Ultra (16 ГБ).** Эмбеддинги для поиска, агрегация, агент-клиент базы. Предлагаемая модель: Qwen3-8B Q4_0 на Hexagon; нужен OpenAI-совместимый API на localhost.
- **Среда на телефоне.** Termux, JupyterLab в браузере телефона, процессы не умирают при выключенном экране.
- **Переносимый сервер.** Jupyter ставится в `jupyter_bin/<win|android>`, данные и база — в `jupyter_data/`; запуск — `start_jupyter.bat` / `start_jupyter.sh`.

**Вне рамок:** облачные модели, доступ из интернета, многопользовательский режим.

## Принятые решения

1. **Запись от ИИ** — сразу в базу, без черновиков; откат — через git (автокоммиты).
2. **Клиенты MCP** — и локальная модель, и внешние агенты (Claude Code с ПК); под разную силу моделей — профили `lite` / `std` / `full`.
3. **Поиск** — для начала по точным подстрокам и regex в памяти; семантический (эмбеддинги) отложен.
4. **Обслуживание** — внешний индексатор (агент) по своему расписанию; плагин только даёт ему API `/mcp/kb-admin`.
5. **Индексатор** — в первую очередь **сильная внешняя модель** (Claude Code и подобные), а не локальная LLM телефона. Правила работы — на сервере, в `jupyter_data/mcp_docs/<lang>/admin.ipynb`: их получает любой агент на `/mcp/kb-admin` (`instructions` + описания команд), правятся в JupyterLab. Для Claude Code — тонкий навык `kb-index`, который только запускает проход и отсылает к правилам сервера.
6. **Полномочия индексатора** — структурные правки сразу (деление и склейка секций, заголовки, `id`, порядок, шапки, теги); смысл и факты не меняет. Удаление содержимого, статей и тем, перенос между статьями — только предложением в отчёте человеку.
7. **Объём и источники** — темы небольшие, списки отдаются целиком; импорта извне пока нет.
8. **Синхронизация** между ПК и телефоном — вне плагина, пока не нужна.
9. **Клиенты с особенностями MCP** — проблемные возможности отключаются параметром подключения `off` (например, `off=structured` для Claude Code).

Открытых вопросов нет.

## Отложено на будущее

- **Связанные теги с релевантностью** (для улучшенного профиля): по тегу — список связанных тегов с уровнем связи (по совместной встречаемости в секциях, позже — по эмбеддингам). Требует полей релевантности у тегов.
- **Семантический поиск** по эмбеддингам от локальной LLM.
- Перенос и переименование статей/тем, склейка дублей — в API обслуживания.
- Ссылки между статьями как отдельная сущность — пока обычные Markdown-ссылки в тексте.
- Запуск на телефоне — следующий шаг проекта.

## Статус

На 2026-11-21:
- Цели и решения согласованы, открытых вопросов нет.
- Плагин реализован в `kb_mcp/`, 27 тестов в `tests/` проходят; проверен вживую на Windows (`start_jupyter.bat`, порт 8888).
- **Публичный репозиторий** (git в корне проекта, ветка `main`): проект назван Ztare AI Local KB MCP, лицензия Unlicense, пометка «написано ИИ». Корневой `.gitignore` пропускает из `jupyter_data` только `mcp_docs`, конфиг сервера, настройки Lab и эту тему (`kb/development/mcp-kb`).
  - Правки статей, уже есть в репозитории, коммитятся обычно (`git add`), в том числе при запущенном Jupyter.
  - **Новые** статьи в `development/mcp-kb` внешний git не видит из-за `kb/.git` автокоммита: остановить Jupyter, удалить `jupyter_data/kb/.git`, `git add -A`, закоммитить.
- Скрипты запуска сокращены до минимума: токен, вывод адресов и команд `claude mcp add` — в плагине (`kb_mcp/launch.py`), ip/порт/браузер — в `jupyter_server_config.json` (`setup#launch`).
- Метаданные ядра: ведёт JupyterLab; плагин только заполняет пустые значениями ядра при записи через MCP / индексатор, чтобы статьи открывались без вопроса о сохранении (`save-prompts`, `storage#format`).
- MCP-серверы в Claude Code: `kb` (профиль `full`, `lang=ru`, `off=structured`) и `kb-admin` (`lang=ru`, `off=structured`).
- Индексатор: правила — `jupyter_data/mcp_docs/<lang>/admin.ipynb`; навык `kb-index` (очередь + уборка тегов, шапок, ссылок, структуры) — статья `development/mcp-kb/skills/kb-index`; агент собирает из неё `SKILL.md` сам, установлен в `~/.claude/skills/kb-index`. Первый проход выполнен 2026-11-20; повторный не зацикливается.
- `put_article` / `put_sections`: `title` и `content` можно не передавать — остаётся текущий текст секции (правка только тегов).
- `version` статьи считается только по значимому содержимому (`admin-api#version`) — пустое сохранение в JupyterLab не даёт `conflict`.
- Документация перенесена из `GOALS.md` и `TZ.md` в эту тему. Ссылки в коде (`kb_mcp/`) указывают на пути статей этой темы (`KB <статья>#<секция>`).

## Карта документации

| Статья | О чём |
|---|---|
| `development/mcp-kb/overview` | эта статья: цели, принципы, режимы, решения, статус |
| `development/mcp-kb/install` | чистая установка на новое место: что нужно, какие файлы перенести, порядок запуска |
| `development/mcp-kb/connect` | как подключить базу к ИИ в другом проекте и перенести туда его документацию |
| `development/mcp-kb/remote` | база на телефоне, доступ с ПК: SSH-туннель, adb, локальная сеть |
| `development/mcp-kb/save-prompts` | как избавиться от вопроса «сохранить?» на нетронутых статьях: настройки Lab и что делает плагин |
| `development/mcp-kb/storage` | хранение: термины, дерево и имена, формат, метаданные секций, теги, жизненный цикл правок, git |
| `development/mcp-kb/api` | API потребителя `/mcp/kb`: пути, команды, профили, настройки подключения, ошибки |
| `development/mcp-kb/answers` | формат ответов MCP: плоский текст, блоки, поля секции, записи и ошибки |
| `development/mcp-kb/admin-api` | API обслуживания `/mcp/kb-admin`: команды индексатора, `version` |
| `development/mcp-kb/plugin` | устройство плагина `kb_mcp/`: модули, ContentsManager, хуки, транспорт, `mcp_docs` |
| `development/mcp-kb/setup` | запуск, токен, подключение клиентов, настройки Lab, тесты |
| `development/mcp-kb/skills/…` | навыки Claude Code (источник правды, агент ставит их сам): `kb-index` — индексатор и уборка |

Правило: прежде чем менять код плагина, обнови соответствующую статью (сначала цели и решения, потом реализация).